# CTDenoiser — Colab Setup
Run each cell top-to-bottom. The repo lives on Google Drive so
weights survive session restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = 'https://github.com/tsereda/CTDenoiser.git'
BRANCH    = 'claude/init-project-setup-vWHBI'
REPO_ROOT = '/content/drive/MyDrive/CTDenoiser'

if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    os.makedirs(REPO_ROOT, exist_ok=True)
    !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    print('Cloned.')
else:
    print('Repo already exists, pulling latest.')

%cd {REPO_ROOT}
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}

In [ ]:
%cd /content/drive/MyDrive/CTDenoiser
!pip install -q -r requirements.txt

In [ ]:
%cd /content/drive/MyDrive/CTDenoiser
!pytest -q

In [ ]:
# Smoke run on synthetic data (no real CT needed)
%cd /content/drive/MyDrive/CTDenoiser
!python -m ctdenoiser.train --model ctformer --epochs 1

In [ ]:
# --- Real data (TCIA LDCT-and-Projection-data) ---
# Build ldct_cache.h5 once with your TCIA download/preprocessing notebook
# (keys: <pid>_low / <pid>_full, shape (num_slices, H, W), raw HU).
# Split is by patient; validation uses full-slice overlapped inference.
import os, shutil

DRIVE_CACHE = '/content/drive/MyDrive/ldct_cache.h5'
LOCAL_CACHE = '/content/ldct_cache.h5'

# Local copy = much faster I/O than reading the .h5 off Drive
if os.path.exists(DRIVE_CACHE) and not os.path.exists(LOCAL_CACHE):
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    print('Copied cache to local Colab disk.')
assert os.path.exists(LOCAL_CACHE), (
    'ldct_cache.h5 not found - run your TCIA preprocessing notebook first.'
)

# Adjust --epochs / --batch-size to your GPU budget.
%cd /content/drive/MyDrive/CTDenoiser
!python -m ctdenoiser.train \
    --model ctformer \
    --h5-cache /content/ldct_cache.h5 \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64

# Baseline for comparison:
# !python -m ctdenoiser.train --model redcnn --h5-cache /content/ldct_cache.h5 --epochs 50 --batch-size 16